### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heloc",
    dataset_year="2021",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/averkiyoliabev/home-equity-line-of-creditheloc",
    download_description="""
We download the data from Kaggle and uzip it to a predefined folder.

mkdir -p local-data-warehouse/heloc/ && cd local-data-warehouse/heloc && kaggle datasets download averkiyoliabev/home-equity-line-of-creditheloc && cd ../../ && unzip local-data-warehouse/heloc/home-equity-line-of-creditheloc.zip -d local-data-warehouse/heloc/ && rm local-data-warehouse/heloc/home-equity-line-of-creditheloc.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{averkiyoliabev2021heloc,
  author       = {Kaggle User Averkiyoliabev},
  title        = {Home Equity Line of Credit (HELOC)},
  year         = {2021},
  howpublished = {\url{https://www.kaggle.com/datasets/averkiyoliabev/home-equity-line-of-creditheloc}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="averkiyoliabev2021heloc",
    license="Public",
    data_tags=["IID"],
    curation_comments="""
- Anomaly: the dataset has time-related features. However, the task and features are preprocessed to be time-invariant.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="RiskPerformance",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="RiskPerformance",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/heloc_dataset_v1 (1).csv")

cat_features = [
    "RiskPerformance"
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,459
Columns: 24
Use sampling: False (sample size: 10,459)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['MSinceOldestTradeOpen', 'AverageMInFile', 'NetFractionInstallBurden', 'NetFractionRevolvingBurden', 'MSinceMostRecentTradeOpen', 'PercentInstallTrades', 'PercentTradesWBalance', 'NumTotalTrades', 'MSinceMostRecentDelq', 'NumSatisfactoryTrades']
Rows remaining as candidates after top-10 filter: 590 (of 10,459)

#### Duplicate Report
Total duplicate rows: 587 (5.61% of dataset)
Duplicate rows ignoring target: 588 (5.62% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,MaxDelqEver,NumTotalTrades,NumTradesOpeninLast12M,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,Bad,69,148,4,66,41,0,0,100,-7,7,8,41,4,10,-7,1,1,32,60,7,3,1,50
1,Bad,77,229,3,109,23,0,0,100,-7,7,8,23,2,35,0,0,0,38,93,4,3,1,58
2,Bad,58,46,7,38,13,0,0,93,8,4,6,5,1,50,-7,2,2,80,84,5,4,1,90
3,Bad,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9,-9
4,Bad,80,226,2,66,35,0,0,100,-7,7,8,36,2,47,0,0,0,2,77,5,7,0,62


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,RiskPerformance,category,0.0,0.0,2.0,"Bad, Good"
1,ExternalRiskEstimate,int64,0.0,0.0,61.0,"-9, 65, 66, 68, 73, 72, 70, 63, 75, 69"
2,MSinceOldestTradeOpen,int64,0.0,0.0,526.0,"-9, -8, 178, 132, 176, 150, 165, 183, 158, 169"
3,MSinceMostRecentTradeOpen,int64,0.0,0.0,112.0,"2, 3, 4, 5, 1, 6, -9, 7, 8, 9"
4,AverageMInFile,int64,0.0,0.0,237.0,"-9, 79, 71, 74, 68, 80, 75, 84, 63, 70"
5,NumSatisfactoryTrades,int64,0.0,0.0,74.0,"-9, 18, 15, 16, 22, 19, 13, 14, 21, 20"
6,NumTrades60Ever2DerogPubRec,int64,0.0,0.0,19.0,"0, 1, 2, -9, 3, 4, 5, 6, 7, 8"
7,NumTrades90Ever2DerogPubRec,int64,0.0,0.0,17.0,"0, 1, -9, 2, 3, 4, 5, 6, 7, 9"
8,PercentTradesNeverDelq,int64,0.0,0.0,72.0,"100, -9, 96, 97, 95, 94, 93, 92, 88, 90"
9,MSinceMostRecentDelq,int64,0.0,0.0,87.0,"-7, -9, 1, 2, 3, 4, 5, -8, 6, 8"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
ExternalRiskEstimate,10459.0,67.425758,21.121621,-9.0,94.0
MSinceOldestTradeOpen,10459.0,184.205373,109.683816,-9.0,803.0
MSinceMostRecentTradeOpen,10459.0,8.543455,13.301745,-9.0,383.0
AverageMInFile,10459.0,73.843293,38.782803,-9.0,383.0
NumSatisfactoryTrades,10459.0,19.428052,13.004327,-9.0,79.0
NumTrades60Ever2DerogPubRec,10459.0,0.042738,2.513910,-9.0,19.0
NumTrades90Ever2DerogPubRec,10459.0,-0.142843,2.367397,-9.0,19.0
PercentTradesNeverDelq,10459.0,86.661536,25.999584,-9.0,100.0
MSinceMostRecentDelq,10459.0,6.762406,20.501250,-9.0,83.0
MaxDelq2PublicRecLast12M,10459.0,4.928291,3.756275,-9.0,9.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                    
RiskPerformance 1      Bad   5459  52.19
                2     Good   5000  47.81

In [8]:
# Target Distribution
target_df

,count,pct
RiskPerformance,,
Bad,5459,52.19
Good,5000,47.81


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to heloc/019d5a64-d939-7d49-bb24-54f75fe34648
019d5a64-d939-7d49-bb24-54f75fe34648
f4f7c553ab16cdf8ac7a77f5545e99c867b31838a05bab7795e63afa8e839e54
